# Causal MAS Distillation - end-to-end Colab run book

This notebook runs the whole pipeline in order. Every cell is independent and
idempotent: re-running a cell that already produced its output is cheap because
all API calls are cached to Google Drive.

**The research question.** At a strictly matched training-token budget, does
supervised fine-tuning on multi-agent debate transcripts (Arm B) produce a
better small model than supervised fine-tuning on plain rejection-sampled
correct solutions (Arm A)?

| Phase | What it does | Cost | Runtime |
|---|---|---|---|
| 0 | Free A/B smoke test from cached data | $0 | ~6 GPU-h |
| 1 | Regenerate traces with the v3 debate harness | ~$3 | ~40 min |
| 2 | The real A/B experiment | $0 | ~6 GPU-h |
| 3 | CRN causal attribution (only if Arm B wins) | ~$15 | ~3 h |

**Runtime type.** Sections 1-4 need CPU only. Sections 5-6 need a T4 GPU
(Runtime -> Change runtime type -> T4 GPU).

---
## 0. Read this before running anything

1. **Every cache path lives on Drive.** Colab local disk is wiped when the
   session closes. This project already lost a probe cache that way and had to
   pay for it twice.
2. **Three values must be identical across every script: `model`, `base_url`,
   `max_tokens`.** They are part of the cache key. A mismatch does not error;
   it silently re-generates everything and charges you.
3. **Do not edit `src/debate/prompts.py` casually.** Prompt strings are part of
   the cache key. Changing one invalidates every cached generation.


---
## 1. Setup


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
CACHE_DIR = '/content/drive/MyDrive/cmd'
pathlib.Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
print('cache dir ready:', CACHE_DIR)
print(sorted(os.listdir(CACHE_DIR)))


Mounted at /content/drive
cache dir ready: /content/drive/MyDrive/cmd
['cache_debates.jsonl', 'cache_debates_v2.jsonl', 'cache_probe.jsonl', 'ckpt']


In [2]:
%cd /content
![ -d causal-mas-distill ] || git clone https://github.com/Arshia-HZ/causal-mas-distill.git
%cd /content/causal-mas-distill
!git pull --ff-only || true
!pip -q install -r requirements-api.txt


/content
Cloning into 'causal-mas-distill'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (259/259), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 259 (delta 104), reused 234 (delta 79), pack-reused 0 (from 0)
Receiving objects: 100% (259/259), 182.90 KiB | 1.34 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/causal-mas-distill
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 8.1 MB/s eta 0:00:00


In [3]:
import os
os.makedirs("data/magdi", exist_ok=True)
!gdown 1yo7sb9b44JpwyDB0OddaxwzS6CS1-bP7 -O data/magdi/MATH_1000.json

Downloading...
From: https://drive.google.com/uc?id=1yo7sb9b44JpwyDB0OddaxwzS6CS1-bP7
To: /content/causal-mas-distill/data/magdi/MATH_1000.json
100% 21.7M/21.7M [00:00<00:00, 65.3MB/s]


In [4]:
import json

data = json.load(open("data/magdi/MATH_1000.json"))

def disagreed(item, r=0):
    votes = item.get(f"vote_{r}") or []
    return len(set(str(v) for v in votes)) > 1

pool = []
for i, item in enumerate(data):
    if not disagreed(item):
        continue
    q = (item.get("question") or "").strip()
    g = str(item.get("gold_answer", "")).strip()
    if q and g:
        pool.append({"pid": f"math_{i:04d}", "question": q, "gold": g})

print(f"disagreement pool: {len(pool)} of {len(data)}")
with open("data/math_problems.json", "w") as f:
    json.dump(pool, f, indent=2)


disagreement pool: 785 of 1000


### API key

Put `GC_API_KEY` in the Colab secrets panel (the key icon in the left sidebar)
and enable notebook access. Never paste the key into a cell you might commit.


In [6]:
from google.colab import userdata
import os
os.environ['GC_API_KEY'] = 'gc__nQfUEeoedn9cMsdx69ximQB0ffZjhLa'
os.environ['API_URL']    = 'https://api.generalcompute.com/v1'
os.environ['MODEL']      = 'deepseek-v3.2'
os.environ['MAX_TOKENS'] = '1024'
print('key loaded:', bool(os.environ['GC_API_KEY']))


key loaded: True


---
## 2. Difficulty probe (already done - skip unless starting fresh)

Samples the teacher 32x per problem to measure a per-problem pass rate.
Produces `data/probed_all.json`, which everything downstream depends on.

Measured result on 785 MATH problems: **629 solved 32/32 (ceiling), 22 solved
0/32 (floor), 134 in between.** 80% of MATH is at teacher ceiling, which is why
every result must be reported split by difficulty.

These 25,120 completions are also your Arm A training data - see section 4.


In [ ]:
!python scripts/00a_probe_difficulty.py \
  --input data/math_problems.json \
  --output data/gate_problems_k32.json \
  --probed-out data/probed_all.json \
  --api-url $API_URL --api-key $GC_API_KEY --model $MODEL \
  --k 32 --keep-min 0.0 --keep-max 1.0 --max-problems 100000 \
  --cache-path /content/drive/MyDrive/cmd/cache_probe.jsonl


^C


---
## 3. Generate debate traces

**Only re-run this if you are regenerating with the v3 debate harness
(Phase 1).** If `data/traces.jsonl` already exists and was made with the v1
harness, skip to section 4 and do the free Phase 0 read first.

`--protocol debate` uses the rewritten harness: role system prompts, a shared
transcript, a length-capped critic that re-derives before it reviews, and a
unique cache nonce per generation. `--protocol selfrefine` reproduces the
original stateless chain. Run both and report the comparison - that contrast
is itself a result.

Use a **new cache file** for v3. The prompt strings changed, so the old cache
cannot be reused and mixing them corrupts nothing but wastes disk.


In [ ]:
!python scripts/01_generate_debates.py \
  --problems data/trace_targets.json \
  --output data/traces_v3.jsonl \
  --protocol debate \
  --api-url $API_URL --api-key $GC_API_KEY --model $MODEL \
  --max-rounds 3 --n-solutions 3 --max-tokens $MAX_TOKENS \
  --cache-path /content/drive/MyDrive/cmd/cache_debates_v3.jsonl


### 3b. Validate the traces before spending anything else

Free, runs in a second, and has caught real problems every time it was run.
Check in particular:
- **seeds per problem** should be `{3: N}` with no problem at 1 or 2
- **identical round-1 solvers** must be 0. Anything above 0 means the API cache
  collapsed independent seeds onto one draw and the traces are worthless.
- **critic dispute rate**: the v1 harness scored 0.08. If v3 does not move this
  number, the rewrite did not work - stop and diagnose before paying for more.


In [ ]:
!python scripts/00f_check_traces.py \
  --traces data/traces_v3.jsonl \
  --targets data/trace_targets.json \
  --max-tokens 1024

!python scripts/00g_diagnose_signal.py \
  --traces data/traces_v3.jsonl \
  --probed data/probed_all.json \
  --max-tokens 1024


---
## 4. Build the matched A/B datasets

### 4a. Arm A, recovered from the probe cache - zero API calls

The difficulty probe already sampled the teacher 32x per problem. Those
completions *are* rejection samples. This recovers and grades them.

**Do not refactor `LEGACY_SOLVE_PROMPT` inside this script into an import.**
It must reproduce cache keys written under the v1 prompt. If you replace it
with `get_solve_prompt`, every lookup misses and recovery silently drops to 0%.


In [7]:
!python scripts/10_build_arm_a.py \
  --cache-path /content/drive/MyDrive/cmd/cache_probe.jsonl \
  --problems data/probed_all.json \
  --model $MODEL --n-probe 32 --max-tokens 768 \
  --out data/arm_a_pool.jsonl


cache entries : 3147
problems      : 785

RECOVERY
  all chunks hit  : 785
  partial hit     : 0
  no hit          : 0
  recovery rate   : 100.0%

ARM A POOL -> data/arm_a_pool.jsonl
  problems written        : 785
  with >=1 correct sample : 763
  total correct solutions : 22621


### 4b. Match the two arms

This is the scientific core. It intersects the problem sets, counts tokens with
the real Qwen tokenizer, fills both arms round-robin to an identical completion-
token budget, and refuses to write output if the arms differ by more than 1%.

**`--clip-critic-words 200` is not optional on v1 traces.** Measured on the 314
correct v1 traces: median transcript is 25,213 characters, about 8,400 student
tokens, and 92.7% exceed the 4096-token `max_seq_length`. Critic messages are
49.3% of all characters - as much as all three solvers combined. Truncation cuts
the end of the sequence, which is where the final answer is, so an unclipped
Arm B would be trained to ramble and never conclude. That would produce
'B loses' as a pure artefact of sequence length.

On v3 traces the critic is already capped at 200 words, so pass `0`.


In [8]:
!python scripts/11_build_ab_datasets.py \
  --arm-a-pool data/arm_a_pool.jsonl \
  --traces data/traces.jsonl \
  --probed data/probed_all.json \
  --clip-critic-words 200 \
  --max-completion-tokens 3584 \
  --budget-tokens 120000 \
  --outdir data


config.json: 100% 660/660 [00:00<00:00, 3.54MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 26.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 56.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 111MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 142MB/s]
problems with a correct RS solution : 763
problems with a correct debate trace: 123
INTERSECTION (trap T3)              : 123
train problems: 91   eval problems: 32
dropped for exceeding 3584 completion tokens: A=61  B=190

MATCHED-TOKEN DATASETS
arm           n  comp_tokens   mean_tok  problems
A (RS)       68       119988     1764.5        67
B (debate)     43       119716     2784.1        22

token gap: 0.23%

matched-example variant: 43 examples per arm
eval problems -> data/eval_problems.json (32)

DIFFICULTY (trap T7): 0/91 training problems are at teacher ceiling (p=1.0)
Report evaluation split by this. The pooled number hides the only interesting stratum.


---
## 5. Train - needs a T4 GPU

Six runs: 2 arms x 3 seeds. Roughly 40-70 minutes each on a free T4.
Hyperparameters are identical across arms; the **only** thing that varies is the
dataset file. Do not tune one arm.

Three seeds is a hard minimum. LoRA SFT on ~1000 examples has run-to-run
standard deviation of 1-2 accuracy points, so a single-seed 2-point difference
is indistinguishable from noise.

Checkpoints go to Drive. A 12-hour Colab cap will not survive all six runs in
one session, so this cell is written to skip any checkpoint that already exists.


In [11]:
!pip -q install -U -r requirements-train.txt
import torch; print(torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [16]:
!pip uninstall -y torchaudio torchvision

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0


In [18]:
!pip install "trl<0.20.0" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 11.4 MB/s eta 0:00:00


In [20]:
!pip install -U "torchao>=0.16.0" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 54.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [31]:
import os, subprocess, pathlib

CKPT_ROOT = '/content/drive/MyDrive/cmd/ckpt'

for arm in ['a', 'b']:
    for seed in [0]:
        out = f'{CKPT_ROOT}/arm_{arm}'
        done = pathlib.Path(f'{out}/seed{seed}/adapter_model.safetensors')
        if done.exists():
            print(f'skip arm_{arm} seed{seed} - already trained')
            continue

        print(f'=== training arm_{arm} seed{seed} ===', flush=True)

        # Capture the output and errors instead of throwing a generic exception
        process = subprocess.run([
            'python', 'scripts/04_train.py',
            '--dataset', f'data/sft_arm_{arm}.jsonl',
            '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
            '--config', 'configs/student_qwen2.5_1.5b.yaml',
            '--output-dir', out,
            '--seed', str(seed),
        ], capture_output=True, text=True)

        # If it failed, print the exact traceback and stop
        if process.returncode != 0:
            print("\n❌ SCRIPT CRASHED ❌")
            print("--- STDOUT (Progress before crash) ---")
            print(process.stdout)
            print("--- STDERR (Actual Error Traceback) ---")
            print(process.stderr)
            break
        else:
            print(process.stdout)
            print(f"✅ Successfully finished arm_{arm} seed{seed}\n")

    # Break outer loop if inner loop failed
    if process.returncode != 0:
        break

=== training arm_a seed0 ===
Loaded 68 training examples
seed=0 -> /content/drive/MyDrive/cmd/ckpt/arm_a/seed0
Starting training...
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
{'loss': '0.5872', 'grad_norm': '0.001584', 'learning_rate': '0.00015', 'num_tokens': '380', 'mean_token_accuracy': '0.8947', 'epoch': '1.118'}
{'loss': '2.012e-07', 'grad_norm': '5.443e-06', 'learning_rate': '4.028e-05', 'num_tokens': '760', 'mean_token_accuracy': '1', 'epoch': '2.235'}
{'train_runtime': '18.99', 'train_samples_per_second': '10.74', 'train_steps_per_second': '1.422', 'train_loss': '0.2175', 'num_tokens': '1020', 'mean_token_accuracy': '1', 'epoch': '3'}
Training complete. Checkpoints saved to /content/drive/MyDrive/cmd/ckpt/arm_a/seed0

✅ Successfully finished arm_a seed0

=== training arm_b seed0 ===
Loaded 43 training examples
seed=0 -> /content/drive/MyDrive/cmd/ckpt/arm_b/seed0
Starting training...
trainable params: 18,464,768 || all params: 1,562,179,072 

---
## 6. Evaluate and decide

Greedy decoding, graded with `eval.grade.is_correct`, paired bootstrap over
problems, and the same table repeated split by teacher difficulty.

**Commit to the decision rule before you look at the output:**

| Result | Meaning | Next |
|---|---|---|
| CI entirely above 0 | Deliberation carries transferable signal | Phase 3, attribution |
| CI spans 0, `|delta| < 0.02` | Null | Phase 1, fix the harness, repeat |
| CI entirely below 0 | Transcripts are worse training data | Real finding against the STaR assumption |

The non-ceiling stratum is the one that matters. 80% of MATH is at teacher
ceiling and both arms will saturate there, so the pooled number hides the
effect in either direction.


In [33]:
!python scripts/12_eval_ab.py \
  --eval data/eval_problems.json \
  --arm-a /content/drive/MyDrive/cmd/ckpt/arm_a/seed0/final \
  --arm-b /content/drive/MyDrive/cmd/ckpt/arm_b/seed0/final \
  --probed data/probed_all.json \
  --out results/ab_eval.json


eval problems: 32

[A] /content/drive/MyDrive/cmd/ckpt/arm_a/seed0/final
W0813 09:24:20.713000 16853 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0813 09:24:21.014000 16853 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 306.28it/s]
  20/32

[B] /content/drive/MyDrive/cmd/ckpt/arm_b/seed0/final
Loading weights: 100% 338/338 [00:01<00:00, 298.16it/s]
  20/32

PER-SEED ACCURACY
arm  seed   acc
A    0      0.156
B    0      0.125

[POOLED]  n=32
  A (rejection sampling) : 0.156
 

### 6b. The matched-example-count variant

At equal tokens, Arm A gets roughly 6x more examples because a transcript is
6x longer than a solution. That is the honest primary comparison, but it
confounds *content* with *number of gradient updates*. Re-run sections 5 and 6
against `sft_arm_a_eqn.jsonl` / `sft_arm_b_eqn.jsonl` to separate the two.

If the two variants disagree, report both. Do not pick the flattering one.


---
## 7. Phase 3 - CRN causal attribution (only if Arm B wins)

Which messages carry the advantage.

**Run the placebo check first.** It runs the same world twice with no ablation.
Under common random numbers the difference must be exactly 0.0 in every world,
because identical prompts hit the cache and return identical text. If it is not
exactly zero, the nonces are wrong and every attribution number is noise.


In [ ]:
import asyncio, json
from src.backends.api import ApiBackend
from src.debate.schema import Trace
from src.counterfactual.crn import placebo_check

traces = [Trace.from_dict(d) for d in json.load(open('data/traces.jsonl'))]
backend = ApiBackend(model='deepseek-v3.2',
                     base_url='https://api.generalcompute.com/v1',
                     api_key_env='GC_API_KEY',
                     cache_path='/content/drive/MyDrive/cmd/cache_crn.jsonl',
                     max_tokens=1024)

print(asyncio.get_event_loop().run_until_complete(
    placebo_check(traces[0], backend, k=8)))


In [ ]:
!python scripts/02_counterfactual_replay.py \
  --traces data/traces.jsonl \
  --output results/utilities_crn.json \
  --api-url $API_URL --api-key $GC_API_KEY --model $MODEL \
  --estimand total_crn --k 32 --trace-concurrency 6 \
  --cache-path /content/drive/MyDrive/cmd/cache_crn.jsonl


---
## 8. Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `10_build_arm_a.py` reports 0% recovery | prompt string or `max_tokens` does not match what the probe sent | try `--max-tokens 768` and `1024`; confirm `LEGACY_SOLVE_PROMPT` is byte-identical to the v1 string |
| `Max_len exceeded: Input is 8357 tokens but this model only supports 8192` | transcript grew past the teacher context | the v3 harness shrinks the transcript and retries; on v1, cap the critic |
| per-problem scores are all 0/3 or 3/3 | API cache collapsed the seeds | check `identical round-1 solvers` in `00f`; the v3 harness adds a unique nonce per generation |
| `11_build_ab_datasets.py` exits with 'arms differ by more than 1%' | one arm ran out of candidate examples before the budget | lower `--budget-tokens` until both arms saturate |
| Arm B accuracy near zero and outputs never terminate | transcripts truncated at `max_seq_length` | pass `--clip-critic-words 200` and `--max-completion-tokens 3584` |
| Colab disconnects mid-training | 90-minute idle timeout / 12-hour cap | checkpoints are on Drive; re-run the training cell, it skips finished seeds |
